PREDICTING STARTUP SURVIVAL: A MACHINE LEARNING PERSPECTIVE
 
> In questa tesi vengono combinate tecniche di Data Mining e Machine Learning
con l’obiettivo di identificare i principali fattori di rischio per le startup (con un
particolare focus su Competitors, Team e Funding), in modo da prevederne l’eventuale fallimento.

> I dati presi in esame provengono dalla piattaforma PitchBook, e
sono stati elaborati in linea con la letteratura economica recente.

> A partire dalle metriche calcolate, sono stati addestrati diversi modelli (Alberi
Decisionali, Random Forest, Reti Neurali Artificiali), i quali sono stati successivamente valutati in base alla capacità di classificare correttamente le startup fallite.
Questi esperimenti hanno permesso, inoltre, di cogliere i principali indicatori di "sopravvivenza" delle startup, mediante il valore dell’importanza che ogni modello
assegna a una determinata feature. Un ulteriore studio è stato svolto tramite l’impiego del Random Survival Forest, una versione alternativa del Random Forest
ideata per l’analisi della sopravvivenza.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from dotenv import load_dotenv 
import yaml

import pandas as pd
import polars as pl
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler,RobustScaler
from sklearn.impute import SimpleImputer,KNNImputer
from src.preprocessing import processUniversityList, getFlagTop50Institute,lump_categories
from imblearn.over_sampling import SMOTE


from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from src.models.MLP import MLP
from sklearn.neural_network import MLPClassifier



from sklearn.metrics import (accuracy_score,confusion_matrix,  f1_score, fbeta_score, precision_score, recall_score,
                             classification_report, roc_curve, auc,
                             precision_recall_curve, average_precision_score, roc_auc_score)
from sklearn.inspection import permutation_importance
import wandb
import shap
import matplotlib.pyplot as plt # Necessario per salvare i grafici
import seaborn as sns

import scipy.integrate
if not hasattr(np, 'trapz'):
    np.trapz = scipy.integrate.trapezoid

load_dotenv()

with open('config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

Lettura della versione iniziale del dataset "master" 

In [ ]:
df = pl.read_csv(config['paths']['raw_dataset'],null_values=["NA"])

print(f'Colonne: {len(df.columns)}') 
print(f'Righe: {len(df)}')

# Sostituisco "Stay" con il valore attuale di "GrowthStageGroup"
df = df.with_columns(
    pl.when(pl.col("GrowthNextStageGroup") == "Stay")
    .then(pl.col("GrowthStageGroup"))
    .otherwise(pl.col("GrowthNextStageGroup"))
    .alias("GrowthNextStageGroup")
)

Pre-processing
Il dataset di partenza è un panel in cui ogni azienda presenta un record per ogni anno di vita disponibile.
Nell' ottica di sperimentare con modelli di classificazione, si è scelto di focalizzarsi sui dati delle aziende che si trovano in fase di Early per provare a prevederne la condizione al qT-esimo anno successivo. Per fare ciò, il dataset viene raggruppato per azienda, mantenendo i valori delle features all'entrata nella fase Early; la variabile target viene invece mantenuta al T-esimo anno successivo o all'ultimo dato disponibile.   

Per evitare troppa diversità tra le aziende sono stati applicati alcuni filtri:
 - Il valore di Age all'entrata della fase Early deve essere <=2 (è così per la maggior parte delle aziende a parte alcuni outliers che vengono così eliminati)
 - L'anno di fondazione deve essere compreso tra il 2010 e il 2024-T (i dati si fermano al 2024)


In [ ]:
T=7

lastYear=2024-T

df_preseed = df.filter(
    pl.col("GrowthStageGroup")
    .eq("Early")
    .any()
    .over("CompanyID")
)
df_preseed=df_preseed.drop_nulls(subset=['GrowthStageGroup'])


df_preseed = df_preseed.select(["CompanyID", "Age"])

# aggregazioni per azienda
df_combined = (
    df_preseed
    .group_by("CompanyID")
    .agg([
        pl.col("Age").min().alias("StartingAge"),
        pl.col("Age").max().alias("LastAge"),
    ])
    .with_columns(
        pl.min_horizontal(
            pl.col("StartingAge") + T,
            pl.col("LastAge")
        ).alias("TargetAge")
    )
    .filter(pl.col("StartingAge") <= 2) # filtro aggiunto per evitare valori troppo diversi (in ogni caso la maggior parte delle aziende aveva un Age compreso tra 0 e 2 al momento dell'entrata in preseed)
)

df = df.join(df_combined, on="CompanyID")


df_target = df.filter(
    pl.col("Age") == pl.col("TargetAge")
).select(["CompanyID", "StartingAge" ,"TargetAge" ,'GrowthStageGroup','TimeNextStageGroup','GrowthNextStageGroup'])


df_target = df_target.with_columns(
    pl.when(pl.col("TargetAge") + pl.col("TimeNextStageGroup")<=pl.col("StartingAge") + T) 
    .then(pl.col("GrowthNextStageGroup"))
    .otherwise(pl.col("GrowthStageGroup"))
    .alias("Target")
)

df_target_panel=df.join(df_target.select(["CompanyID", "Target"]),on='CompanyID')


df_target_final = df_target_panel.filter(
    (pl.col("Age") == pl.col("StartingAge")) &
    (pl.col("YearFounded")+pl.col("Age") <= lastYear) &
    (pl.col("YearFounded")+pl.col("Age") >= 2010)
)




Feature selection

In [ ]:
df_target_final=df_target_final.select(['CompanyID', 'Target' ,'YearFounded','Age', 'N_Deal', 'TotalRaised_Est', 
                                        'Percent_Females', 'Is_Eco', 'Is_Eng', 'Is_NS', 
                                        'Is_Hum', 'Is_SS', 'Is_Med', 'Is_Law', 'Is_IT', 
                                         'Institute', 'WorkExp_Idx_Mean', 'Total_Founders', 'Is_Debt', 
                                         'Is_SpinOff', 'Is_CrowdFunding',  
                                        'MeanMedianRoundAmount_cum', 'Is_Accelerator', 'has_Corporate', 
                                        'has_VentureCapital', 'has_PublicInvestor', 'has_Angel_Lead', 'has_Corporate_Lead',
                                        'has_VentureCapital_Lead', 'has_Accelerator_Lead', 'has_PrivateEquity_Lead', 'has_PublicInvestor_Lead', 
                                          'HQCountry', 'PrimaryIndustrySector',
                                        'SimilarityScoreMean', 'N_Competitors', 'Same_Country','Highest_Degree_CEO','Gender_CEO','MeanTotalInvestments_cum',
                                        'WorkExperienceIndex_CEO', 'Is_Angel',  'Total_People', 'Is_Grant','has_PrivateEquity','TotalInvestors', 'Highest_Degree_Mean','Avg_Earliest_Year'
                                        ])
                                       
 # 'MeanTotalActivePortfolio_cum'


Viene creata la feature booleana HasTop50Institute che ha valore positivo se almeno una delle Università citate nella colonna 'Institutes' è presente nella classifica delle 50 migliori università al mondo. La classifica è stata stilata da QS World University Rankings ed è contenuta nel file QS_World_Rankings.csv.

Per fare ciò è stato svolto un confronto di stringhe tramite Jaccard Similarity.

In fase di preprocessing e di match sono state utilizzate diverse tecniche di NLP:

> Text Normalization: 
    Prima del confronto, il codice "pulisce" il testo per ridurre la variabilità semantica. Tutto il testo viene convertito in minuscolo. Vengono rimossi caratteri speciali come virgole, trattini e spazi multipli tramite Espressioni Regolari. Il codice rimuove parole comuni che non aiutano a distinguere un'università dall'altra, come "university", "of", "the". In NLP, queste sono considerate "stop words" specifiche del dominio accademico.

> Nella funzione calculate_similarity_by_words, il codice trasforma le stringhe in insiemi di parole: La stringa viene spezzata in singole unità (parole) usando lo spazio come delimitatore (Tokenizzazione). Trasformando la lista in un set(), il codice ignora l'ordine delle parole e le ripetizioni, concentrandosi solo sulla presenza dei termini (Bag of Words).

> Viene implementata una logica di Named Entity Disambiguation basata sugli acronimi: Tramite la Regex r"\((.*?)\)" viene estratto il testo tra parentesi (es. "MIT" da "Massachusetts Institute of Technology (MIT)"). Il confronto avviene su due binari: se l'acronimo combacia, l'istituto è confermato immediatamente, riducendo i falsi negativi dovuti a nomi troppo lunghi o complessi.

> Logica di Matching Ibrida: Il confronto finale non è una semplice uguaglianza, ma un sistema a cascata: Exact Match su Acronimo: Se l'acronimo estratto è presente nella lista Top 50. Exact Match su Stringa Pulita: Se la stringa normalizzata è identica. Substring Matching: Verifica se un acronimo è contenuto all'interno del nome (es: "MIT" in "MIT Boston"). Fuzzy Matching (Soglia): Se la similarità di Jaccard supera il threshold=0.5.

In [ ]:
# Caricamento e pulizia università
top_50 = processUniversityList(config['paths']['raw_university_ranking'])

df_target_final=df_target_final.with_columns(pl.col("Institute").map_elements( lambda x: getFlagTop50Institute(x, top_50),return_dtype=pl.Boolean, 
        skip_nulls=False).alias("HasTop50Institute"))

df_target_final=df_target_final.drop("Institute")

counts = df_target_final["HasTop50Institute"].value_counts()

Rimozione nazioni non abbastanza frequenti e frequency encoding della colonna 'HQCountry'

Frequency encoding colonna 'PrimaryIndustryGroup'

In [ ]:
df_target_final = lump_categories(df_target_final, "HQCountry", min_count=1000)


#frequency encoding HQCountry
freq_df = df_target_final.group_by("HQCountry").agg(
    pl.len().alias("HQCountryFreq")
)

df_target_final = df_target_final.join(freq_df, on="HQCountry").drop("HQCountry")


#frequency encoding PrimaryIndustrySector

freq_df = df_target_final.group_by("PrimaryIndustrySector").agg(
    pl.len().alias("PrimaryIndustrySectorFreq")
)

df_target_final = df_target_final.join(freq_df, on="PrimaryIndustrySector").drop("PrimaryIndustrySector")


# feature indicatore per il genere del CEO
df_target_final = df_target_final.to_dummies("Gender_CEO").drop("Gender_CEO_Male").drop("Gender_CEO_null")

Gestione Missing Values

Scarto le righe che hanno missing valueas per Total_People, in quanto presentano lo stesso problema per la maggior parte delle features del team.
Conversione delle variabili booleane in numeri interi, revisione tipi variabili numeriche.

In [ ]:
df_target_final = df_target_final.drop_nulls(subset=['Total_People'])


df_target_final = df_target_final.with_columns(
    pl.col(pl.Boolean).cast(pl.Int8)
)

# #Per queste variabili si è deciso di imputare i missing values a 0
variabili_toInt = [ 'N_Competitors','Same_Country'] 

df_target_final = df_target_final.with_columns(
    [pl.col(name).cast(pl.Int64).fill_null(0) for name in variabili_toInt]

)

df_target_final = df_target_final.with_columns(
    pl.col('SimilarityScoreMean').fill_null(pl.col('SimilarityScoreMean').mean()) 

)

#Conversione mantenendo i missing values 
df_target_final = df_target_final.with_columns(
    pl.col(["YearFounded","Age",'Total_Founders','Total_People','Is_Eco','Is_Eng','Is_NS','Is_Hum','Is_SS','Is_Med','Is_Law','Is_IT']).cast(pl.Int64)
)

Missing values per feature

In [ ]:
# 1. Definiamo la soglia (40%)
threshold = 0.4

# 2. Identifichiamo le colonne da tenere
# Calcoliamo la frazione di null per ogni colonna e filtriamo i nomi
cols_to_keep = [
    col for col in df_target_final.columns 
    if df_target_final[col].null_count() / len(df_target_final) < threshold
]

# 3. Sovrascriviamo il dataframe (o creiamone uno nuovo)
df_target_final = df_target_final.select(cols_to_keep)

In [ ]:
result = (
    df_target_final.select([
        (pl.col(col).null_count() / pl.len() * 100).alias(col)
        for col in df_target_final.columns
    ])
    .unpivot(variable_name="colonna", value_name="percentuale_missing")
    .filter(pl.col("percentuale_missing") > 0)
    .sort("percentuale_missing", descending=True)
)

print(result)

Missing values per i record

In [ ]:
# 1. Conta i null in modo orizzontale (molto più veloce e sicuro)
df_with_row_counts = df_target_final.with_columns(
    null_count_row = pl.sum_horizontal(pl.all().is_null())
)

# 2. Raggruppa per vedere la distribuzione
distribuzione_missing = (
    df_with_row_counts
    .group_by("null_count_row")
    .agg(pl.len().alias("numero_di_righe"))
    .sort("null_count_row")
)

print(distribuzione_missing)

In [ ]:
iniziale=len(df_target_final)
# Drop delle righe con 4 o più missing values
print(f"Dataset prima del drop: {iniziale} righe")

df_target_final = df_with_row_counts.filter(pl.col("null_count_row") < 4).drop("null_count_row")

finale=len(df_target_final)
print(f"Righe rimosse: {iniziale - finale}")


Grafico bilanciamento Target

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Calcolo delle frequenze direttamente dal dataset
target_counts = df_target_final['Target'].value_counts()

# Estraiamo etichette e valori in ordine
labels = target_counts["Target"].to_list()
sizes = target_counts["count"].to_list()

# 2. Definizione colori (Palette 'Set1' di Matplotlib, molto simile alla tua immagine)
# Se hai più di 4 categorie, questa palette si adatta automaticamente
colors = plt.get_cmap('Set1').colors

# 3. Creazione del grafico
fig, ax = plt.subplots(figsize=(8, 8))

# autopct calcola la percentuale reale dai dati
patches, texts, autotexts = ax.pie(
    sizes, 
    labels=labels, 
    autopct='%1.1f%%', 
    colors=colors, 
    startangle=140,
    wedgeprops={'linewidth': 1.5, 'edgecolor': 'white'},
    textprops={'fontsize': 11}
)

# Rendiamo le percentuali interne più leggibili (bianche o nere a seconda del gusto)
for autotext in autotexts:
    autotext.set_color('black')
    autotext.set_weight('bold')

# 4. Estetica finale
plt.axis('equal') 

plt.tight_layout()
plt.show()

Creazione dataset per la classificazione binaria

In [ ]:
dataset = df_target_final.with_columns(
    pl.when(pl.col("Target").is_in(["Later", "Exit"]))
    .then(1)
    .otherwise(0)
    .alias("Target")
)

dataset.write_csv(config['paths']['dataset'])


Correlazione features

In [ ]:
dataset = pd.read_csv(config["paths"]["dataset"])
corr_matrix = dataset.drop(['CompanyID', 'Target'],axis=1).corr()


# plt.figure(figsize=(30,20))
# ax = sns.heatmap(data = corr_matrix,cmap='YlGnBu',annot=True)

# bottom, top = ax.get_ylim()
# ax.set_ylim(bottom + 0.5,top - 0.5)


# Trasforma la matrice in una serie di coppie (stack)
# e rimuove i duplicati (unstack della parte superiore della matrice)
sol = (corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
                  .stack()
                  .reset_index())

sol.columns = ['Variabile_1', 'Variabile_2', 'Correlazione']

# Applica il filtro per i range richiesti
mask = (
    ((sol['Correlazione'] >= 0.90) & (sol['Correlazione'] <= 0.99)) |
    ((sol['Correlazione'] <= -0.90) & (sol['Correlazione'] >= -0.99))
)

risultato = sol[mask].sort_values(by='Correlazione', ascending=False)

print(risultato)





Creazione sweep (INIZIO ESPERIMENTO)

In [ ]:
dataset = pd.read_csv(config["paths"]["dataset"])

entity=os.getenv("entity")
project=os.getenv("project")

# Inizializza lo sweep
sweep_id = wandb.sweep(config["sweep_settings"],entity=entity, project=project)

def to_tensors(X, y):
                y_np = y.values if hasattr(y, 'values') else y
                return (
                    torch.tensor(X, dtype=torch.float32),
                    torch.tensor(y_np, dtype=torch.float32).unsqueeze(1)
                )
def get_probs(loader, model, device):
                all_probs, all_labels = [], []
                with torch.no_grad():
                    for X_batch, y_batch in loader:
                        X_batch = X_batch.to(device)
                        logits = model(X_batch)
                        probs = torch.sigmoid(logits).cpu().numpy()
                        all_probs.extend(probs)
                        all_labels.extend(y_batch.numpy())
                return np.array(all_probs).flatten(), np.array(all_labels).flatten().astype(int)


def find_best_threshold(y_true, probs, beta=1, min_accuracy=None):
    """
    Trova la soglia che massimizza l'Fbeta score.
    beta=1 → F1, beta=2 → F2 (più peso al recall)
    min_accuracy → se specificato, scarta soglie che non lo rispettano
    """
    thresholds = np.arange(0.1, 0.9, 0.01)
    best_t, best_score = 0.5, 0
    for t in thresholds:
        preds_t = (probs >= t).astype(int)
        if min_accuracy is not None:
            if accuracy_score(y_true, preds_t) < min_accuracy:
                continue
        score = fbeta_score(y_true, preds_t, beta=beta, zero_division=0)
        if score > best_score:
            best_score = score
            best_t = t
    return best_t, best_score


ids = dataset['CompanyID']  # Salva la colonna ID
X = dataset.drop(['CompanyID', 'Target'],axis=1)
y = dataset['Target']

worst_features = []

# Split: 60% train, 20% validation (threshold tuning), 20% test (final evaluation)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=config['test_size'], stratify=y, random_state=config['random_seed'])
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=config['random_seed'])

imputer = KNNImputer(n_neighbors=5)
X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(X_test)


scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled   = scaler.transform(X_val_imp)
X_test_scaled  = scaler.transform(X_test_imp)


# --- SMOTE (commentato — si usa pos_weight nella loss per bilanciare) ---
# smote = SMOTE(random_state=config['random_seed'])
# X_res, y_res = smote.fit_resample(X_train_scaled, y_train)


print(f"Total: {len(y)} | Train: {len(y_train)} | Validation: {len(y_val)} | Test: {len(y_test)}")

In [ ]:
def train():
    with wandb.init():
        wandb_config = wandb.config
        
        # --- RANDOM FOREST ---
        if wandb_config.model_type == "rf":
            model = RandomForestClassifier(
                n_estimators=wandb_config.n_estimators,
                max_depth=wandb_config.max_depth, 
                min_samples_leaf=wandb_config.min_samples_leaf,
                max_features='log2',
                # min_samples_split=wandb_config.min_samples_split,
                class_weight='balanced',  
                random_state=config['random_seed']
            )

            model.fit(X_train_imp, y_train)
            probs_train = model.predict_proba(X_train_imp)[:, 1]
            probs_val = model.predict_proba(X_val_imp)[:, 1]
            probs_test = model.predict_proba(X_test_imp)[:, 1]

            labels_train = y_train
            labels_val = y_val
            labels_test = y_test

        # --- LGBM ---
        elif wandb_config.model_type == "lgb":
            # Calcolo del rapporto per scale_pos_weight (80/20 -> 4)
            ratio = float(y_train.value_counts()[0] / y_train.value_counts()[1]) 
            
            model = lgb.LGBMClassifier(
                n_estimators=wandb_config.n_estimators, 
                learning_rate=wandb_config.learning_rate,
                max_depth=wandb_config.max_depth,
                scale_pos_weight=ratio,  
                min_child_weight=wandb_config.min_child_weight, 
                subsample=wandb_config.subsample,               
                colsample_bytree=wandb_config.colsample_bytree,
                reg_alpha=wandb_config.reg_alpha,
                reg_lambda=wandb_config.reg_lambda,
                random_state=config['random_seed']
            )

            model.fit(X_train_imp, y_train)
            probs_train = model.predict_proba(X_train_imp)[:, 1]
            probs_val = model.predict_proba(X_val_imp)[:, 1]
            probs_test = model.predict_proba(X_test_imp)[:, 1]

            labels_train = y_train
            labels_val = y_val
            labels_test = y_test

        # --- DECISION TREE ---
        elif wandb_config.model_type == "dt":
            model = DecisionTreeClassifier(
                max_depth=wandb_config.max_depth,
                min_samples_leaf=wandb_config.min_samples_leaf,
                min_samples_split=wandb_config.dt_min_samples_split,
                criterion=wandb_config.dt_criterion,
                class_weight='balanced',
                random_state=config['random_seed']
            )

            model.fit(X_train_imp, y_train)
            probs_train = model.predict_proba(X_train_imp)[:, 1]
            probs_val = model.predict_proba(X_val_imp)[:, 1]
            probs_test = model.predict_proba(X_test_imp)[:, 1]

            labels_train = y_train
            labels_val = y_val
            labels_test = y_test

        # --- LOGISTIC REGRESSION (baseline) ---
        elif wandb_config.model_type == "lr":
            model = LogisticRegression(
                C=wandb_config.lr_C,
                penalty=wandb_config.lr_penalty,
                solver='saga',
                class_weight='balanced',
                max_iter=1000,
                random_state=config['random_seed']
            )

            model.fit(X_train_scaled, y_train)
            probs_train = model.predict_proba(X_train_scaled)[:, 1]
            probs_val = model.predict_proba(X_val_scaled)[:, 1]
            probs_test = model.predict_proba(X_test_scaled)[:, 1]

            labels_train = y_train
            labels_val = y_val
            labels_test = y_test

        elif wandb_config.model_type == "mlp":
            # ── 1. IPERPARAMETRI DA SWEEP ─────────────
            hidden_sizes = [int(x) for x in wandb_config.hidden_sizes.split(",")]
            mlp_lr = wandb_config.mlp_learning_rate
            mlp_dropout = wandb_config.dropout_rate
            mlp_wd = wandb_config.weight_decay
            BATCH_SIZE = wandb_config.batch_size

            # ── 2. CONVERSIONE IN TENSORI ─────────────
            X_train_t, y_train_t = to_tensors(X_train_scaled, y_train)
            X_val_t,   y_val_t   = to_tensors(X_val_scaled,   y_val)
            X_test_t,  y_test_t  = to_tensors(X_test_scaled,  y_test)

            # ── 3. DATALOADER ─────────────────────────
            train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                                    batch_size=BATCH_SIZE, shuffle=True)
            val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t),
                                    batch_size=BATCH_SIZE)
            test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t),
                                    batch_size=BATCH_SIZE)

            # ── 4. MODELLO, LOSS, OPTIMIZER ───────────
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

            model = MLP(
                input_size=X_train_scaled.shape[1],
                hidden_sizes=hidden_sizes,
                dropout_rate=mlp_dropout,
                batch_norm=True
            ).to(device)

            # Class weight: pos_weight = n_neg / n_pos (bilanciamento tramite loss)
            n_pos = y_train.sum()
            n_neg = len(y_train) - n_pos
            pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

            optimizer = torch.optim.AdamW(model.parameters(), lr=mlp_lr, weight_decay=mlp_wd)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='max', factor=0.5, patience=5
            )
            # ── 5. TRAINING LOOP ──────────────────────
            EPOCHS = 100
            best_auc = 0.0
            patience_counter = 0
            EARLY_STOPPING_PATIENCE = 10

            for epoch in range(EPOCHS):

                # ── Train ──
                model.train()
                train_loss = 0.0

                for X_batch, y_batch in train_loader:
                    X_batch, y_batch = X_batch.to(device), y_batch.to(device)

                    optimizer.zero_grad()
                    preds = model(X_batch)
                    loss  = criterion(preds, y_batch)
                    loss.backward()
                    optimizer.step()

                    train_loss += loss.item() * len(X_batch)

                train_loss /= len(train_loader.dataset)

                # ── Validation ──
                model.eval()
                val_loss = 0.0
                all_preds, all_labels = [], []

                with torch.no_grad():
                    for X_batch, y_batch in val_loader:
                        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

                        logits = model(X_batch)
                        loss   = criterion(logits, y_batch)
                        val_loss += loss.item() * len(X_batch)

                        probs = torch.sigmoid(logits).cpu().numpy()
                        all_preds.extend(probs)
                        all_labels.extend(y_batch.cpu().numpy())

                val_loss /= len(val_loader.dataset)
                val_auc   = roc_auc_score(all_labels, all_preds)
                val_f1    = f1_score(all_labels, (np.array(all_preds) > 0.5).astype(int))

                scheduler.step(val_auc)

                print(f"Epoch {epoch+1:3d} | "
                    f"Train Loss: {train_loss:.4f} | "
                    f"Val Loss: {val_loss:.4f} | "
                    f"Val AUC: {val_auc:.4f} | "
                    f"Val F1: {val_f1:.4f}")

                # ── Early Stopping + Best Model ──
                if val_auc > best_auc:
                    best_auc = val_auc
                    patience_counter = 0
                    torch.save(model.state_dict(), "best_model.pt")
                else:
                    patience_counter += 1
                    if patience_counter >= EARLY_STOPPING_PATIENCE:
                        print(f"\nEarly stopping at epoch {epoch+1}. Best AUC: {best_auc:.4f}")
                        break
            # ── 6. VALUTAZIONE ────────────────────────
            model.load_state_dict(torch.load("best_model.pt", weights_only=True))
            model.eval()

            probs_train, labels_train = get_probs(train_loader, model, device)
            probs_val,   labels_val   = get_probs(val_loader,   model, device)
            probs_test,  labels_test  = get_probs(test_loader,  model, device)

        # Trova la soglia ottimale sul validation set con fbeta (beta=2 per dare più peso al recall, importante in questo contesto) e con vincolo di accuracy >= 0.75
        # best_threshold, best_fbeta_val = find_best_threshold(labels_val, probs_val, beta=1, min_accuracy=0.70)

        best_threshold = 0.5
        best_fbeta_val=0

        # Applicazione soglia ottimale
        preds_test  = (probs_test  >= best_threshold).astype(int)
        preds_train = (probs_train >= best_threshold).astype(int)

        # Metriche sul test set
        acc       = accuracy_score(labels_test, preds_test)
        acc_train = accuracy_score(labels_train, preds_train)
        precision = precision_score(labels_test, preds_test, zero_division=0)
        recall    = recall_score(labels_test, preds_test, zero_division=0)
        f1        = f1_score(labels_test, preds_test, zero_division=0)
        f1_train  = f1_score(labels_train, preds_train, zero_division=0)

        # ROC Curve (test set)
        fpr, tpr, _ = roc_curve(labels_test, probs_test)
        roc_auc = auc(fpr, tpr)

        fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
        ax_roc.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
        ax_roc.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
        ax_roc.set_xlim([0.0, 1.0])
        ax_roc.set_ylim([0.0, 1.05])
        ax_roc.set_xlabel('False Positive Rate')
        ax_roc.set_ylabel('True Positive Rate')
        ax_roc.set_title(f'ROC - {wandb_config.model_type.upper()}')
        ax_roc.legend(loc='lower right')
        ax_roc.grid(alpha=0.3)

        # Precision-Recall Curve (test set)
        prec_curve, rec_curve, _ = precision_recall_curve(labels_test, probs_test)
        ap = average_precision_score(labels_test, probs_test)

        fig_pr, ax_pr = plt.subplots(figsize=(8, 6))
        ax_pr.plot(rec_curve, prec_curve, color='green', lw=2, label=f'PR curve (AP = {ap:.3f})')
        ax_pr.set_xlim([0.0, 1.0])
        ax_pr.set_ylim([0.0, 1.05])
        ax_pr.set_xlabel('Recall')
        ax_pr.set_ylabel('Precision')
        ax_pr.set_title(f'Precision-Recall - {wandb_config.model_type.upper()}')
        ax_pr.legend(loc='upper right')
        ax_pr.grid(alpha=0.3)

        # Log delle metriche
        wandb.log({
            "accuracy_train": acc_train,
            "F1_train": f1_train,
            "accuracy": acc,
            "F1": f1,
            "precision": precision,
            "recall": recall,
            "AUC": roc_auc,
            "average_precision": ap,
            "best_threshold": best_threshold,
            "Fbeta_val_at_threshold": best_fbeta_val,
            "roc_curve": wandb.Image(fig_roc),
            "pr_curve": wandb.Image(fig_pr),
        })
        plt.close(fig_roc)
        plt.close(fig_pr)

        print(f"Best threshold (val set): {best_threshold:.2f} | Fbeta val: {best_fbeta_val:.3f}")
        print(f"\nTest set results (threshold={best_threshold:.2f}):")
        print(classification_report(labels_test, preds_test))
        print(f"AUC: {roc_auc:.3f} | AP: {ap:.3f}")


        wandb.sklearn.plot_confusion_matrix(labels_test, preds_test, ["Neg", "Pos"])
        if(True):
            # Dati coerenti con il modello usato
            if wandb_config.model_type in ["rf", "lgb", "dt"]:
                explainer_sample = X_test.iloc[:1000]          # non scaled — RF, LGBM, DT fittati su X_train_imp
            elif wandb_config.model_type == "lr":
                explainer_sample = X_test_scaled[:1000]         # scaled — LR fittata su X_train_scaled
            else:
                explainer_sample = X_test_scaled[:1000]         # scaled — MLP fittato su X_train_scaled

            if wandb_config.model_type in ["rf", "lgb", "dt"]:
                explainer = shap.TreeExplainer(model)
                shap_result = explainer.shap_values(explainer_sample)

                # LGBM e RF moderni restituiscono array 3D (n_samples, n_features, n_classes)
                # oppure lista [classe_0, classe_1] — gestiamo entrambi
                if isinstance(shap_result, list):
                    shap_values = shap_result[1]
                elif hasattr(shap_result, 'shape') and len(shap_result.shape) == 3:
                    shap_values = shap_result[:, :, 1]
                else:
                    shap_values = shap_result

            elif wandb_config.model_type == "lr":
                # Logistic Regression — LinearExplainer (esatto e veloce per modelli lineari)
                explainer = shap.LinearExplainer(model, X_train_scaled)
                shap_values = explainer.shap_values(explainer_sample)

            else:
                # MLP — KernelExplainer con forward pass PyTorch
                def mlp_predict(x):
                    model.eval()
                    with torch.no_grad():
                        t = torch.tensor(x, dtype=torch.float32).to(device)
                        logits = model(t)
                        probs = torch.sigmoid(logits).cpu().numpy().flatten()
                    return probs

                background = shap.sample(X_train_scaled, 30)   # background da dati scaled (coerente con MLP)
                explainer = shap.KernelExplainer(mlp_predict, background)
                shap_values = explainer.shap_values(explainer_sample, nsamples=1000)
                # KernelExplainer con output 1D restituisce già un array 2D — nessuna gestione extra

            # Summary plot — uguale per tutti
            plt.figure(figsize=(10, 6))
            shap.summary_plot(
                shap_values,
                explainer_sample,
                feature_names=X.columns.tolist(),
                show=False
            )
            wandb.log({"shap_summary_plot": wandb.Image(plt)})
            plt.close()

            # Le 10 feature meno importanti secondo SHAP
            # mean_abs_shap = np.abs(shap_values).mean(axis=0)

            # shap_importance = pd.DataFrame({
            #     'feature': X.columns.tolist(),
            #     'importance': mean_abs_shap
            # }).sort_values('importance', ascending=True)  # ascending=True → le peggiori prime

            # bottom_10 = shap_importance.head(10)['feature'].tolist()
            # worst_features.extend(bottom_10)  # Salva le feature da rimuovere per eventuali iterazioni future
            # print("Feature da rimuovere:", bottom_10)

In [ ]:
wandb.agent(sweep_id, function=train, count=100)